<a href="https://colab.research.google.com/github/jelvalojel-glitch/ML_task_02/blob/main/task02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Шаг 1: Создание или загрузка данных и общий осмотр
Для демонстрации сгенерируем синтетический датасет, содержащий как числовые, так и категориальные признаки, а также целевую переменную.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Генерируем случайные данные для демонстрации
np.random.seed(42)
n_samples = 1000

data = {
    'age': np.random.normal(35, 10, n_samples).astype(int), # числовой
    'income': np.random.exponential(50000, n_samples),      # числовой с выбросами/асимметрией
    'experience': np.random.randint(0, 15, n_samples),     # числовой (дискретный)
    'education': np.random.choice(['School', 'Bachelor', 'Master', 'PhD'], size=n_samples, p=[0.2, 0.5, 0.2, 0.1]), # категориальный
    'gender': np.random.choice(['Male', 'Female', 'Unknown'], size=n_samples, p=[0.48, 0.48, 0.04]), # категориальный (с редкой категорией)
    'target': np.random.choice([0, 1], size=n_samples, p=[0.85, 0.15]) # целевой класс (дисбаланс)
}

df = pd.DataFrame(data)
# Добавим немного пропусков
df.loc[df.sample(frac=0.05).index, 'income'] = np.nan

# Общий осмотр
print("--- Первые 5 строк датасета ---")
display(df.head())

print("\n--- Общая информация ---")
df.info()

### Шаг 2: Классификация признаков
Определим типы признаков программно.

In [ ]:
numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_features.remove('target') # убираем целевую переменную
categorical_features = df.select_dtypes(include=[object, 'category']).columns.tolist()

print(f"Числовые признаки: {numeric_features}")
print(f"Категориальные признаки: {categorical_features}")
print(f"Целевая переменная: target")

### Шаг 3: Анализ числовых признаков (распределение и выбросы)

In [ ]:
fig, axes = plt.subplots(len(numeric_features), 2, figsize=(14, 4 * len(numeric_features)))

for i, col in enumerate(numeric_features):
    # Гистограмма распределения
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i, 0], color='skyblue')
    axes[i, 0].set_title(f'Распределение {col}')

    # Boxplot для поиска выбросов
    sns.boxplot(x=df[col].dropna(), ax=axes[i, 1], color='lightcoral')
    axes[i, 1].set_title(f'Диаграмма размаха (выбросы) {col}')

plt.tight_layout()
plt.show()

### Шаг 4: Анализ категориальных признаков (частоты и редкие категории)

In [ ]:
for col in categorical_features:
    plt.figure(figsize=(8, 4))
    # Подсчет частот
    counts = df[col].value_counts(dropna=False)
    percentages = df[col].value_counts(normalize=True, dropna=False) * 100

    print(f"\nЧастоты для признака '{col}':")
    for idx in counts.index:
        print(f"  {idx}: {counts[idx]} ({percentages[idx]:.2f}%)")

    sns.countplot(data=df, x=col, palette='pastel')
    plt.title(f'Частотное распределение для {col}')
    plt.show()

### Шаг 5: Корреляционная матрица и диаграммы рассеяния

In [ ]:
# Строим корреляционную матрицу
plt.figure(figsize=(8, 6))
corr_matrix = df[numeric_features].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Корреляционная матрица числовых признаков')
plt.show()

# Матрица диаграмм рассеяния (Pairplot)
sns.pairplot(df[numeric_features + ['target']], hue='target', palette='Set1')
plt.suptitle('Диаграммы рассеяния для числовых признаков', y=1.02)
plt.show()

### Шаг 6: Оценка дисбаланса целевой переменной

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='target', palette='Set2')
plt.title('Распределение целевой переменной')
plt.show()

target_counts = df['target'].value_counts()
target_pct = df['target'].value_counts(normalize=True) * 100
print(f"Класс 0: {target_counts[0]} ({target_pct[0]:.2f}%)")
print(f"Класс 1: {target_counts[1]} ({target_pct[1]:.2f}%)")